# Survival Analysis for LGD model 

Various steps used by banks to derive the loss rate for unresolved cases 

Historical average 
Advance techniques such as Survival technique 
Cap the facilities to a certain amount based on the historical data  
conservattive approach , unresolved will have maximum loss at the time of default 


Calculate all the defaulted cases for suvival model . This has to be done for all defaults only since it is a LGD model and would consider below specific points :
1. Currently under recovery process
2. Dataset is for unsecured retail customers
3. Banks want to know about the survival probability of the defaulter customers which are not part of the LGD model
   

Steps involved :

1. Pre processing for Survival dataset
2. Understanding the dataset
3. Apply the algorithm based on the dataset description 
4. filter for obervations which have recovery vector 
5. for the max recovery time for all observations and select the time to inlcude maximum observations
6. Only non negative cash flows and recover rates are inlcuded 


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import functions as F
from functools import reduce 
#from pyspark.sql.functions import when,col,avg

# Logistic Regression 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler

# for visual representation , understanding data 

import matplotlib.pyplot as plt 
import seaborn as sns 



In [2]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("survival").getOrCreate()

In [4]:
spark

In [3]:
os.chdir(r'C:\Users\prashant\Modelling\Data')


In [4]:
# for the recovery amount we would be using default dataset 
dd = spark.read.option('header',True)\
               .option('delimiter',',')\
               .option('inferSchema',True)\
               .csv(r'C:\Users\prashant\Modelling\Data\default_data.csv')

In [5]:
# get the main loan dataset
lnd = spark.read.option('header',True)\
                .option('delimiter',',')\
                .option('inferSchema',True)\
                .csv(r'C:\Users\prashant\Modelling\Data\loan_Data.csv')

In [6]:
# get the main LGD dataset
lgd = spark.read.option('header',True)\
                .option('delimiter',',')\
                .option('inferSchema',True)\
                .csv(r'C:\Users\prashant\Modelling\Data\loss_Data.csv')

In [7]:
# to assess the shape of dataframe 
def spark_shape(self):
    return(self.count(),len(self.columns))
pyspark.sql.dataframe.DataFrame.shape = spark_shape

In [8]:
#dd = dd.filter(F.col('default_time') == 1)
dd = dd.filter(F.col('default_time') == 1)\
         .withColumn('EAD',F.col('balance_time'))

In [9]:
# check for final default cases
dd = dd.select(['id'\
                ,'first_time'\
                ,'mat_time'\
                ,'balance_time'\
                ,'LTV_time'\
                ,'interest_rate_time'\
                ,'rate_time'\
                ,'hpi_time'\
                ,'gdp_time'\
                ,'balance_orig_time'\
                ,'default_time'\
                ,'EAD'])

In [10]:
lgd = lgd.filter(F.col('default_time') == 1)\
         .withColumn('EAD',F.col('balance_time'))

join default dataset with loss dataset to get the recoveries , to assess the defaults for which recovery process is still underway this is needed since the recoveries dataset will be added back for Cox regression post estimation of coeffecients using ulitmate or partial recoveries dataset

Features used for final product :
check ones which are binary 
1. Mortgage/product month when defaulted
2. Original month/term 
3. LTV
4. Int rate/int_rate
5. hpi
6. gdp
7. FICO
8. ID
9. Original Balance/loan_amnt 
10. Left Balance/total_pymnt - 



From Loan dataset : 

1. Prod_m_def
2. term
3. balance_time
4. LTV(concatenate with Default data and fill the values)
5. int_rate 
6. hpi
7. gdp
8. fico(concatenate with Default data and fill the values)
9. id
10. funded_amnt
11. total_rec_prncp + total_rec_int + total_rec_late_fee + recoveries



In [11]:
# For no recovery cases LGD would be 1 , thereore post joining the LGD data and DD data need to fill the missing vaulues with 1
cols_dd = ['id','first_time','mat_time','balance_time','balance_orig_time','EAD','hpi_time','gdp_time','LTV_time','interest_rate_time','default_time']
dd_lgd_ead = dd.filter(F.col('default_time') == 1)\
               .join(lgd,dd['id'] == lgd['id'],'left')\
               .select(*[dd[col] for col in cols_dd],lgd['lgd_time'].alias('LGD'))


In [12]:
dd_lgd_ead = dd_lgd_ead.withColumnRenamed('interest_rate_time','int_rate_time')

In [15]:
dd_lgd_ead.where(dd_lgd_ead.LGD.isNull()).count()

304

In [13]:
# Filling up null LGD values with 1 since there are all non recovery cases therefore total loss for the bank 
lgd_fill = 1
dd_lgd_ead = dd_lgd_ead.fillna({'LGD' : lgd_fill})


In [152]:
dd_lgd_ead.printSchema()


root
 |-- id: integer (nullable = true)
 |-- first_time: integer (nullable = true)
 |-- mat_time: integer (nullable = true)
 |-- balance_time: double (nullable = true)
 |-- balance_orig_time: double (nullable = true)
 |-- EAD: double (nullable = true)
 |-- hpi_time: double (nullable = true)
 |-- gdp_time: double (nullable = true)
 |-- LTV_time: double (nullable = true)
 |-- int_rate_time: double (nullable = true)
 |-- default_time: integer (nullable = true)
 |-- LGD: double (nullable = true)



## To get LGD and EAD for all defaulted cases 

In [14]:
# data processing for lnd 
lnd = lnd.withColumn("term", F.regexp_replace("term", "months", ""))

# Data Manipulations to Loan dataset 

In [15]:
# check for missing values in mths_since_last_delinq for default cases and replace  the missing with mths_since_last_record 
# since there is no data on vintage age of the portfolio
lnd = lnd.filter(F.col('loan_status') == 'Default')\
         .withColumn('mths_since_last_delinq',F.when(F.col('mths_since_last_delinq').isNull(),F.col('mths_since_last_record')).\
         otherwise(F.col('mths_since_last_delinq')))

In [16]:
 lnd.filter(F.col('mths_since_last_delinq').isNull()).count()

360

In [17]:
lnd = lnd.withColumn("mths_since_last_delinq", lnd["mths_since_last_delinq"].cast("float"))
lnd = lnd.withColumn("term", lnd["term"].cast("float"))  
lnd = lnd.withColumn("mths_since_last_delinq", lnd["mths_since_last_delinq"].cast("float"))

In [18]:
lnd.columns

['_c0',
 'id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'acc_now_delinq',
 'tot_coll_amt',
 '

# understanding data ditribution for finalized vectors 

In [ ]:
# QQ plots

# Data distribution  visual Test -Step 1 
# Creating QQ plots 
fig, axs = plt.subplots(1, 4, figsize=(18, 7))
stats.probplot(combined_df_charts['N_o2c'], dist="norm", plot=axs[0])
axs[0].set_title('QQ Plot for NDX Open to close ')
stats.probplot(combined_df_charts['N_h2l'], dist="norm", plot=axs[1])
axs[1].set_title('QQ Plot for NDX high to low')
stats.probplot(combined_df_charts['T_o2c'], dist="norm", plot=axs[2])
axs[2].set_title('QQ Plot for Tesla Open to close')
stats.probplot(combined_df_charts['T_h2l'], dist="norm", plot=axs[3])
axs[3].set_title('QQ Plot for Tesla high to low')

# checking these option for Missing data replacement:
1. Minimum of vactor
2. Maximum of vector
3. Average of Vector
4. MICE
5. KNN
6. 
 



# filling up the missing values of mths_since_last_delinq  with the average/min/max values , the selection of the values will be 
# based on best fit measures 
m_del_avg = lnd.select(F.mean(F.col('mths_since_last_delinq'))).first()[0]
m_del_min = lnd.select(F.min(F.col('mths_since_last_delinq'))).first()[0]
m_del_max = lnd.select(F.max(F.col('mths_since_last_delinq'))).first()[0]
m_del_med = lnd.approxQuantile('mths_since_last_delinq',[0.50],0.01)[0]

# replace the col values with avereage 
lnd_avg = lnd.fillna({'mths_since_last_delinq' : m_del_avg})

# replace the col values with median 
lnd_med = lnd.fillna({'mths_since_last_delinq' : m_del_med})


# replace the col values with min 
lnd_min = lnd.fillna({'mths_since_last_delinq' : m_del_min})


# replace the col values with max 
lnd_max = lnd.fillna({'mths_since_last_delinq' : m_del_max})



In [23]:
# best value is the average within the threshold of portfolio term requirements therefore using lnd_avg as final dataset 
print(m_del_avg,m_del_min,m_del_max,m_del_med)

41.529661016949156 1.0 118.0 36.0


In [24]:
# rounding up
lnd_avg = lnd_avg.withColumn('mths_since_last_delinq',F.ceil(lnd_avg.mths_since_last_delinq))

In [25]:
lnd_avg.shape()

(832, 75)

In [ ]:
# dd = dd.select(['id','first_time','mat_time','balance_time','LTV_time','interest_rate_time','hpi_time','gdp_time','balance_orig_time','default_time'])

In [128]:
dd = dd.withColumnRenamed('interest_rate_time','int_rate_time')

In [129]:
lnd_avg = lnd_avg.withColumn('balance_time',(lnd_avg.total_rec_prncp + lnd_avg.total_rec_int + lnd_avg.total_rec_late_fee + lnd_avg.recoveries))

In [130]:
lnd_avg = lnd_avg.withColumn('LTV_time',F.lit(0))
lnd_avg = lnd_avg.withColumn('hpi_time',F.lit(0))
lnd_avg = lnd_avg.withColumn('gdp_time',F.lit(0))
lnd_avg = lnd_avg.withColumn('default_time',F.lit(1))


In [132]:
lnd_avg = lnd_avg.withColumnRenamed('funded_amnt','balance_orig_time').withColumnRenamed('int_rate','int_rate_time')
lnd_avg = lnd_avg.withColumnRenamed('mths_since_last_delinq','first_time').withColumnRenamed('term','mat_time')
lnd_avg = lnd_avg.withColumn('EAD',F.col('balance_time'))

In [163]:
lnd_avg = lnd_avg.withColumn('LGD',(F.col('balance_orig_time') - F.col('balance_time'))/F.col('EAD'))

In [164]:
lnd_avg = lnd_avg.select(['id'\
                          ,'first_time'\
                          ,'mat_time'\
                          ,'balance_time'\
                          ,'LTV_time'\
                          ,'int_rate_time'\
                          ,'hpi_time'\
                          ,'gdp_time'\
                          ,'balance_orig_time'\
                          ,'default_time'\
                          ,'EAD'\
                          ,'LGD'])


In [166]:
# Conctenate overall default and loan default dataset for processed dataset that will be used for Survival Analysis
df_list = [lnd_avg,dd_lgd_ead]
df = reduce(F.DataFrame.unionByName,df_list)

In [167]:
df.select(df.columns[:20]).show(3)

+-------+----------+--------+------------+--------+-------------+--------+--------+-----------------+------------+--------+--------------------+
|     id|first_time|mat_time|balance_time|LTV_time|int_rate_time|hpi_time|gdp_time|balance_orig_time|default_time|     EAD|                 LGD|
+-------+----------+--------+------------+--------+-------------+--------+--------+-----------------+------------+--------+--------------------+
|1062399|        61|    60.0|    19767.48|     0.0|        17.27|     0.0|     0.0|          18000.0|           1|19767.48|-0.08941352160214654|
| 879297|        42|    60.0|    23343.83|     0.0|        14.27|     0.0|     0.0|          21250.0|           1|23343.83| -0.0896952213925479|
| 809235|        79|    60.0|     6643.32|     0.0|        15.99|     0.0|     0.0|           5600.0|           1| 6643.32|-0.15704798203307982|
+-------+----------+--------+------------+--------+-------------+--------+--------+-----------------+------------+--------+-------

In [29]:
# checking for total sum of missing values 
df.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+---+----------+--------+------------+--------+-------------+--------+--------+-----------------+------------+
| id|first_time|mat_time|balance_time|LTV_time|int_rate_time|hpi_time|gdp_time|balance_orig_time|default_time|
+---+----------+--------+------------+--------+-------------+--------+--------+-----------------+------------+
|  0|         0|       0|           0|       1|            0|       0|       0|                0|           0|
+---+----------+--------+------------+--------+-------------+--------+--------+-----------------+------------+



# Steps for Model building 
1. Understand the data distribution 
2. Apply the Survival Function

In [30]:
# converting to pandas to apply the steps below 
dfp = df.toPandas()

Not performing normalization of data since survival doesnt require normalization . Also this data doesnt have continous variables therefore there is no need for one hot encoding . Even though normalization is always good to perform for any model but this is more applicable for regression based models such as Lasso, Ridge and Elastic regression because the penalty coeffecients are same for all variables , meaning they dont change as per the variables error ro distance from the regression line.

First i would be performing Linear and Lgistic Regressions and using Raphson Algorithm to find best coeffcients therefore i am applying Normalization for better coeffecients . For Survival we would be 

In [31]:
dfp.describe()

,id,first_time,mat_time,balance_time,LTV_time,int_rate_time,hpi_time,gdp_time,balance_orig_time,default_time
count,2.357000e+03,2357.000000,2357.000000,2.357000e+03,2356.000000,2357.000000,2357.000000,2357.000000,2.357000e+03,2357.0
mean,6.236791e+06,31.644039,108.482391,1.629932e+05,62.963482,10.599460,113.858265,0.315277,1.660720e+05,1.0
std,1.037519e+07,15.648684,48.140720,1.831538e+05,49.638169,5.019633,86.539066,1.902315,1.792339e+05,0.0
min,9.000000e+00,1.000000,36.000000,4.430400e+02,0.000000,2.000000,0.000000,-4.146711,0.000000e+00,1.0
25%,2.013300e+04,25.000000,60.000000,9.154820e+03,0.000000,7.125000,0.000000,0.000000,1.935000e+04,1.0
50%,3.817400e+04,28.000000,141.000000,1.117962e+05,82.367041,8.875000,154.940000,0.000000,1.147500e+05,1.0
75%,1.055772e+07,41.000000,146.000000,2.520000e+05,105.913932,13.980000,180.520000,1.692969,2.520000e+05,1.0
max,3.783081e+07,118.000000,228.000000,1.518109e+06,155.613265,26.060000,226.290000,4.320114,1.440000e+06,1.0


In [33]:
# manipluate the dataset for LTV , HPI and gdp columns
coltofill = ['LTV_time','hpi_time','gdp_time']
for col in coltofill:
    dfp[col].fillna(dfp[col].mean(),inplace=True)


In [47]:
dfp.describe()

,id,first_time,mat_time,balance_time,LTV_time,int_rate_time,hpi_time,gdp_time,balance_orig_time,default_time,recovery_rate
count,2.357000e+03,2357.000000,2357.000000,2.357000e+03,2357.000000,2357.000000,2357.000000,2357.000000,2.357000e+03,2357.0,2357.000000
mean,6.236791e+06,31.644039,108.482391,1.629932e+05,62.963482,10.599460,113.858265,0.315277,1.660720e+05,1.0,inf
std,1.037519e+07,15.648684,48.140720,1.831538e+05,49.627633,5.019633,86.539066,1.902315,1.792339e+05,0.0,NaN
min,9.000000e+00,1.000000,36.000000,4.430400e+02,0.000000,2.000000,0.000000,-4.146711,0.000000e+00,1.0,0.187920
25%,2.013300e+04,25.000000,60.000000,9.154820e+03,0.000000,7.125000,0.000000,0.000000,1.935000e+04,1.0,0.585449
50%,3.817400e+04,28.000000,141.000000,1.117962e+05,82.348070,8.875000,154.940000,0.000000,1.147500e+05,1.0,0.982302
75%,1.055772e+07,41.000000,146.000000,2.520000e+05,105.910985,13.980000,180.520000,1.692969,2.520000e+05,1.0,0.999520
max,3.783081e+07,118.000000,228.000000,1.518109e+06,155.613265,26.060000,226.290000,4.320114,1.440000e+06,1.0,inf


In [35]:
# make the target variables and perfrom corelation(for linear relationships) and Bayesian Copula for non-linear relationships 
dfp['recovery_rate'] = dfp['balance_time']/dfp['balance_orig_time']

In [54]:
# Survival needs to be done only on positive cash flows between 0 and 1 
dfp_final = dfp[(dfp['recovery_rate'] >= 0) & (dfp['recovery_rate'] <= 1)]

In [56]:
dfp_final.value_counts()

id        first_time  mat_time  balance_time  LTV_time    int_rate_time  hpi_time  gdp_time   balance_orig_time  default_time  recovery_rate
72        33          141.0     127858.64     101.403077  7.000          153.35    -4.146711  129500.0           1             0.987325         1
6522104   42          36.0      12123.18      0.000000    15.880         0.00       0.000000  14400.0            1             0.841888         1
6714785   42          36.0      10569.00      0.000000    11.550         0.00       0.000000  13350.0            1             0.791685         1
6706615   42          60.0      12702.54      0.000000    24.890         0.00       0.000000  18075.0            1             0.702768         1
6704710   42          60.0      13969.98      0.000000    15.880         0.00       0.000000  24000.0            1             0.582082         1
                                                                                                                                 

# Performing Logistic Regession using below of the 2 mthods 

1. Linear Regression : Deriving the LGD with Predicted variables and getting the coeffecients which are best suited
2. Logistic Regerssion : Steps to logistic regression ;
    a.Decide the threshold , ususally between 0.1 to 0.9
    b.Finalize the best beta coeffecient using Newton Raphson algorithm 
    c.Get LGD predictions based on best beta coefficient 

In [12]:
# Normalizing the dataset before applying regression 
lr_minmax = Pipeline([
    ('scaler',MinMaxScaler()),
     ])

In [19]:
lr_standard = Pipeline([
    ('scaler',StandardScaler()),
     ])